In [ ]:
import pandas as pd
import numpy as np

DATA = '../data/individual/processed'

VELOCITY_THRESHOLD = 0.1
FIXATION_ANXIETY_THRESHOLD = 250  # ms, short fixations suggest hypervigilance

def calculate_fixation_duration(df, velocity_threshold=VELOCITY_THRESHOLD):
    df = df.copy()
    df['gaze_velocity'] = (
        np.sqrt(df['gazeDir.x'].diff()**2 +
                df['gazeDir.y'].diff()**2 +
                df['gazeDir.z'].diff()**2) / df['reltime'].diff()
    ).replace([np.inf, -np.inf], np.nan)

    df['is_fixation'] = df['gaze_velocity'] < velocity_threshold
    df['fixation_id'] = (df['is_fixation'] != df['is_fixation'].shift()).cumsum()

    fixation_durations = df[df['is_fixation']].groupby('fixation_id')['reltime'].apply(
        lambda x: x.max() - x.min()
    )
    fixation_durations = fixation_durations[fixation_durations > 0]
    return fixation_durations.mean()

baseline_data = pd.read_csv(f'{DATA}/sed.csv')
session_01_data = pd.read_csv(f'{DATA}/sed_01.csv')
session_02_data = pd.read_csv(f'{DATA}/sed_02.csv')
session_03_data = pd.read_csv(f'{DATA}/sed_03.csv')

baseline_ms = calculate_fixation_duration(baseline_data) * 1000
session_01_ms = calculate_fixation_duration(session_01_data) * 1000
session_02_ms = calculate_fixation_duration(session_02_data) * 1000
session_03_ms = calculate_fixation_duration(session_03_data) * 1000

diff_01 = session_01_ms - baseline_ms
diff_02 = session_02_ms - baseline_ms
diff_03 = session_03_ms - baseline_ms

def anxiety_label(ms):
    return "Anxiety" if ms < FIXATION_ANXIETY_THRESHOLD else "Normal"

print(f'Baseline Average Fixation Duration: {baseline_ms:.2f} ms - {anxiety_label(baseline_ms)}')
print(f'Session 1 Average Fixation Duration: {session_01_ms:.2f} ms - {anxiety_label(session_01_ms)}')
print(f'Difference from Baseline in Session 1: {diff_01:.2f} ms')
print(f'Session 2 Average Fixation Duration: {session_02_ms:.2f} ms - {anxiety_label(session_02_ms)}')
print(f'Difference from Baseline in Session 2: {diff_02:.2f} ms')
print(f'Session 3 Average Fixation Duration: {session_03_ms:.2f} ms - {anxiety_label(session_03_ms)}')
print(f'Difference from Baseline in Session 3: {diff_03:.2f} ms')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA = '../data/individual/processed'

VELOCITY_THRESHOLD = 0.1
FIXATION_ANXIETY_THRESHOLD = 250  # ms, short fixations suggest hypervigilance

def calculate_fixation_durations(df, velocity_threshold=VELOCITY_THRESHOLD):
    df = df.copy()
    df['gaze_velocity'] = (
        np.sqrt(df['gazeDir.x'].diff()**2 +
                df['gazeDir.y'].diff()**2 +
                df['gazeDir.z'].diff()**2) / df['reltime'].diff()
    ).replace([np.inf, -np.inf], np.nan)

    df['is_fixation'] = df['gaze_velocity'] < velocity_threshold
    df['fixation_id'] = (df['is_fixation'] != df['is_fixation'].shift()).cumsum()

    fixation_durations = df[df['is_fixation']].groupby('fixation_id')['reltime'].apply(
        lambda x: x.max() - x.min()
    )
    return fixation_durations[fixation_durations > 0]

baseline_data = pd.read_csv(f'{DATA}/sed.csv')
session_01_data = pd.read_csv(f'{DATA}/sed_01.csv')
session_02_data = pd.read_csv(f'{DATA}/sed_02.csv')
session_03_data = pd.read_csv(f'{DATA}/sed_03.csv')

baseline_ms = calculate_fixation_durations(baseline_data) * 1000
session_01_ms = calculate_fixation_durations(session_01_data) * 1000
session_02_ms = calculate_fixation_durations(session_02_data) * 1000
session_03_ms = calculate_fixation_durations(session_03_data) * 1000

average_durations = [
    baseline_ms.mean(),
    session_01_ms.mean(),
    session_02_ms.mean(),
    session_03_ms.mean()
]
labels = ['Baseline', 'Session 01', 'Session 02', 'Session 03']

plt.figure(figsize=(8, 6))
plt.bar(labels, average_durations, color=['blue', 'orange', 'green', 'red'])
plt.axhline(FIXATION_ANXIETY_THRESHOLD, color='r', linestyle='dashed', linewidth=1, label='Anxiety Threshold (<250 ms)')
plt.ylabel('Average Fixation Duration (ms)')
plt.title('Average Fixation Duration by Dataset')
plt.legend()
plt.show()
plt.close()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

DATA = '../data/individual/processed'
PSY = '../data/individual/psychometric'

question_types_count = {'HADS': 14, 'STAI-T': 20, 'STAI-S': 20, 'BFI': 10, 'FQ': 24}

def plot_fixation_duration_for_category(sed_data, psy_data, category_name, expected_questions, session_num, window_size=20):
    category_data = psy_data[psy_data['Type'] == category_name].copy()

    segments, question_times = [], []
    for _, row in category_data.iterrows():
        mask = (sed_data['datetime'] >= row['Question Start Time']) & (sed_data['datetime'] <= row['Question Answer Time'])
        seg = sed_data.loc[mask]
        if not seg.empty:
            segments.append(seg)
        question_times.append(row['Question Answer Time'])

    category_eye = pd.concat(segments, ignore_index=True) if segments else pd.DataFrame()
    if len(question_times) != expected_questions:
        print(f"Warning: {category_name} has {len(question_times)} questions, expected {expected_questions}")

    category_eye = category_eye.sort_values('datetime').reset_index(drop=True)
    category_eye['smoothed_fixation_duration'] = category_eye['duration'].rolling(window=window_size).mean()
    category_eye = category_eye[np.isfinite(category_eye['smoothed_fixation_duration'])]

    plt.figure(figsize=(12, 6))
    plt.plot(category_eye['datetime'], category_eye['smoothed_fixation_duration'], label='Fixation Duration', color='b')

    for i, time in enumerate(question_times, start=1):
        if not category_eye.empty:
            nearest_idx = (category_eye['datetime'] - time).abs().idxmin()
            fd = category_eye.loc[nearest_idx, 'smoothed_fixation_duration']
            plt.scatter(category_eye.loc[nearest_idx, 'datetime'], fd, color='red', s=50, zorder=5)
            plt.text(category_eye.loc[nearest_idx, 'datetime'], fd + 0.1, f'Q{i}', fontsize=9, rotation=45, ha='right')

    plt.xlabel('Time')
    plt.ylabel('Fixation Duration')
    plt.title(f'Fixation Duration During {category_name} - Session {session_num:02d}')
    plt.xticks(rotation=45)
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    plt.close()

for i in range(1, 4):
    sed = pd.read_csv(f'{DATA}/sed_fix_{i:02d}.csv')
    sed['datetime'] = pd.to_datetime(sed['datetime'], format='%Y/%m/%d %H:%M:%S.%f', utc=True, errors='coerce').dt.tz_convert(None)
    psy = pd.read_csv(f'{PSY}/Psychometric_Test_Results_{i:02d}.csv')
    psy['Question Start Time'] = pd.to_datetime(psy['Question Start Time'], utc=True, errors='coerce').dt.tz_convert(None)
    psy['Question Answer Time'] = pd.to_datetime(psy['Question Answer Time'], utc=True, errors='coerce').dt.tz_convert(None)
    for category, expected in question_types_count.items():
        plot_fixation_duration_for_category(sed, psy, category, expected, i)